In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/captioning_project/data', exist_ok=True)
os.makedirs('/content/drive/MyDrive/captioning_project/models', exist_ok=True)
os.makedirs('/content/drive/MyDrive/captioning_project/outputs', exist_ok=True)


print("Drive mounted. Folders ready.")

In [ ]:
!pip install sentence-transformers nltk efficientnet-pytorch evaluate -q

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')

nltk.download('punkt')

print("Libraries ready.")

In [ ]:
import os, re, json
import numpy as np
import pandas as pd
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from sentence_transformers import SentenceTransformer, util

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

In [ ]:
import os, json

kaggle_username = "RoshniiiChawla"
kaggle_key      = "KGAT_07f08dc7a86358fac1d2fc85ba1dcc6a"

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

# Download to Colab local (faster, more reliable)
print("Downloading to Colab local storage...")
!kaggle datasets download -d hsankesara/flickr-image-dataset \
    -p /content/flickr30k --unzip

print("✓ Download complete!")
print(os.listdir('/content/flickr30k'))

In [ ]:
import os
for root, dirs, files in os.walk('/content/flickr30k'):
    level = root.replace('/content/flickr30k', '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/ ({len(files)} files)")

In [ ]:
import os

base = '/content/drive/MyDrive/captioning_project/data'
for root, dirs, files in os.walk(base):
    level = root.replace(base, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/ ({len(files)} files)")

In [ ]:
import os

IMG_DIR  = '/content/flickr30k/flickr30k_images/flickr30k_images/flickr30k_images'
CAP_FILE = '/content/flickr30k/flickr30k_images/results.csv'
SAVE_DIR = '/content/drive/MyDrive/captioning_project/models/'
OUT_DIR  = '/content/drive/MyDrive/captioning_project/outputs/'

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

assert os.path.exists(IMG_DIR),  f"Images not found at {IMG_DIR}"
assert os.path.exists(CAP_FILE), f"Captions not found at {CAP_FILE}"
print("✓ All paths verified.")
print(f"  Images : {len(os.listdir(IMG_DIR))} files")


In [ ]:
df = pd.read_csv(CAP_FILE, sep='|',
                 names=['image', 'cap_num', 'caption'],
                 skiprows=1)

df['image']   = df['image'].astype(str).str.strip()
df['caption'] = df['caption'].astype(str).str.strip().str.lower()
df['caption'] = df['caption'].apply(lambda x: re.sub(r"[^a-z0-9\s]", "", x))
df['caption'] = '<start> ' + df['caption'] + ' <end>'

# Drop bad rows
df = df.dropna(subset=['caption', 'image'])
df = df[df['caption'].str.len() > 10]

print(f"Total caption rows : {len(df)}")
print(f"Unique images      : {df['image'].nunique()}")
print(df.head(3))

In [ ]:
class Vocabulary:
    def __init__(self, freq_threshold=5):
        self.freq_threshold = freq_threshold
        self.itos = {0: '<pad>', 1: '<start>', 2: '<end>', 3: '<unk>'}
        self.stoi = {v: k for k, v in self.itos.items()}

    def build(self, captions):
        counter = Counter()
        for cap in captions:
            counter.update(cap.split())
        idx = 4
        for word, freq in counter.items():
            if freq >= self.freq_threshold:
                self.itos[idx] = word
                self.stoi[word] = idx
                idx += 1
        print(f"✓ Vocabulary built: {len(self.itos)} words")

    def encode(self, caption):
        return [self.stoi.get(w, self.stoi['<unk>'])
                for w in caption.split()]

    def decode(self, indices):
        return ' '.join([self.itos.get(i, '<unk>')
                         for i in indices
                         if i not in [0, 1, 2]])

    def __len__(self):
        return len(self.itos)

vocab = Vocabulary(freq_threshold=5)
vocab.build(df['caption'].tolist())

# Save vocab to Drive so you never rebuild it
import pickle

# Build vocab fresh
vocab = Vocabulary(freq_threshold=5)
vocab.build(df['caption'].tolist())

# Save to Drive for future sessions
with open(SAVE_DIR + 'vocab.pkl', 'wb') as f:
    pickle.dump(vocab, f)
print(f"✓ Vocab built and saved: {len(vocab)} words")



In [ ]:
class Vocabulary:
    def __init__(self, freq_threshold=5):
        self.freq_threshold = freq_threshold
        self.itos = {0: '<pad>', 1: '<start>', 2: '<end>', 3: '<unk>'}
        self.stoi = {v: k for k, v in self.itos.items()}

    def build(self, captions):
        counter = Counter()
        for cap in captions:
            counter.update(cap.split())
        idx = 4
        for word, freq in counter.items():
            if freq >= self.freq_threshold:
                self.itos[idx] = word
                self.stoi[word] = idx
                idx += 1
        print(f"✓ Vocabulary built: {len(self.itos)} words")

    def encode(self, caption):
        return [self.stoi.get(w, self.stoi['<unk>'])
                for w in caption.split()]

    def decode(self, indices):
        return ' '.join([self.itos.get(i, '<unk>')
                         for i in indices
                         if i not in [0, 1, 2]])

    def __len__(self):
        return len(self.itos)

vocab = Vocabulary(freq_threshold=5)
vocab.build(df['caption'].tolist())

# Save vocab to Drive so you never rebuild it
import pickle
with open(SAVE_DIR + 'vocab.pkl', 'wb') as f:
    pickle.dump(vocab, f)
print("✓ Vocab saved to Drive.")

In [ ]:
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),       # augmentation
    transforms.ColorJitter(0.2, 0.2, 0.2),  # augmentation
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

class Flickr30kDataset(Dataset):
    def __init__(self, df, vocab, img_dir, transform=None, max_len=50):
        self.df        = df.reset_index(drop=True)
        self.vocab     = vocab
        self.img_dir   = img_dir
        self.transform = transform
        self.max_len   = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['image'])
        image    = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        tokens  = self.vocab.encode(row['caption'])
        tokens  = tokens[:self.max_len]
        tokens += [0] * (self.max_len - len(tokens))  # pad

        return image, torch.tensor(tokens, dtype=torch.long)

# 80 / 10 / 10 split by image (not row)
unique_imgs = df['image'].unique()
np.random.seed(42)
np.random.shuffle(unique_imgs)
n = len(unique_imgs)

train_imgs = set(unique_imgs[:int(0.8*n)])
val_imgs   = set(unique_imgs[int(0.8*n):int(0.9*n)])
test_imgs  = set(unique_imgs[int(0.9*n):])

train_df = df[df['image'].isin(train_imgs)]
val_df   = df[df['image'].isin(val_imgs)]
test_df  = df[df['image'].isin(test_imgs)]

train_set = Flickr30kDataset(train_df, vocab, IMG_DIR, transform_train)
val_set   = Flickr30kDataset(val_df,   vocab, IMG_DIR, transform_val)
test_set  = Flickr30kDataset(test_df,  vocab, IMG_DIR, transform_val)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=32, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=32, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Train : {len(train_set):,} samples")
print(f"Val   : {len(val_set):,} samples")
print(f"Test  : {len(test_set):,} samples")

In [ ]:
class EfficientNetEncoder(nn.Module):
    def __init__(self, encoded_size=7):
        super().__init__()
        base = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        self.features       = base.features        # (B, 1280, 7, 7)
        self.adaptive_pool  = nn.AdaptiveAvgPool2d((encoded_size, encoded_size))

        # Freeze first 6 blocks, fine-tune last 2
        for i, block in enumerate(self.features):
            for p in block.parameters():
                p.requires_grad = i >= 6

    def forward(self, x):
        x = self.features(x)                        # (B, 1280, 7, 7)
        x = self.adaptive_pool(x)                   # (B, 1280, 7, 7)
        B, C, H, W = x.shape
        x = x.view(B, C, -1).permute(0, 2, 1)      # (B, 49, 1280)
        return x

In [ ]:
class ForegroundAttention(nn.Module):
    def __init__(self, encoder_dim, decoder_dim, attention_dim):
        super().__init__()
        self.enc_att  = nn.Linear(encoder_dim, attention_dim)
        self.dec_att  = nn.Linear(decoder_dim, attention_dim)
        self.full_att = nn.Linear(attention_dim, 1)
        self.relu     = nn.ReLU()
        self.softmax  = nn.Softmax(dim=1)

    def forward(self, encoder_out, decoder_hidden):
        att1  = self.enc_att(encoder_out)                    # (B, 49, att_dim)
        att2  = self.dec_att(decoder_hidden).unsqueeze(1)   # (B, 1,  att_dim)
        att   = self.full_att(self.relu(att1 + att2)).squeeze(2)
        alpha = self.softmax(att)                            # (B, 49)
        ctx   = (encoder_out * alpha.unsqueeze(2)).sum(dim=1)
        return ctx, alpha


class BackgroundAttention(nn.Module):
    """
    Novelty: inverts foreground weights to attend to
    background/minor object regions via a learned gate.
    """
    def __init__(self, encoder_dim, decoder_dim, attention_dim):
        super().__init__()
        self.enc_att  = nn.Linear(encoder_dim, attention_dim)
        self.dec_att  = nn.Linear(decoder_dim, attention_dim)
        self.full_att = nn.Linear(attention_dim, 1)
        self.relu     = nn.ReLU()
        self.softmax  = nn.Softmax(dim=1)
        self.gate     = nn.Sequential(
            nn.Linear(encoder_dim * 2, 1),
            nn.Sigmoid()
        )

    def forward(self, encoder_out, decoder_hidden, fg_alpha):
        att1      = self.enc_att(encoder_out)
        att2      = self.dec_att(decoder_hidden).unsqueeze(1)
        att       = self.full_att(self.relu(att1 + att2)).squeeze(2)
        bg_w      = (1.0 - fg_alpha) * self.softmax(att)
        bg_w      = bg_w / (bg_w.sum(dim=1, keepdim=True) + 1e-8)
        bg_ctx    = (encoder_out * bg_w.unsqueeze(2)).sum(dim=1)
        return bg_ctx, bg_w


class DualAttentionDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, decoder_dim,
                 encoder_dim=1280, attention_dim=512, dropout=0.5):
        super().__init__()
        self.encoder_dim  = encoder_dim
        self.decoder_dim  = decoder_dim
        self.vocab_size   = vocab_size

        self.embedding    = nn.Embedding(vocab_size, embed_dim)
        self.fg_attention = ForegroundAttention(encoder_dim, decoder_dim, attention_dim)
        self.bg_attention = BackgroundAttention(encoder_dim, decoder_dim, attention_dim)
        self.fusion_gate  = nn.Sequential(
            nn.Linear(encoder_dim * 2, encoder_dim),
            nn.Sigmoid()
        )
        self.lstm   = nn.LSTMCell(embed_dim + encoder_dim, decoder_dim)
        self.init_h = nn.Linear(encoder_dim, decoder_dim)
        self.init_c = nn.Linear(encoder_dim, decoder_dim)
        self.fc     = nn.Linear(decoder_dim, vocab_size)
        self.drop   = nn.Dropout(dropout)

    def init_hidden(self, encoder_out):
        m = encoder_out.mean(dim=1)
        return torch.tanh(self.init_h(m)), torch.tanh(self.init_c(m))

    def forward(self, encoder_out, captions):
        B, _, _ = encoder_out.shape
        T       = captions.size(1) - 1
        embs    = self.drop(self.embedding(captions))  # (B, T+1, embed_dim)
        h, c    = self.init_hidden(encoder_out)

        preds     = torch.zeros(B, T, self.vocab_size).to(encoder_out.device)
        fg_alphas = torch.zeros(B, T, encoder_out.size(1)).to(encoder_out.device)
        bg_alphas = torch.zeros(B, T, encoder_out.size(1)).to(encoder_out.device)

        for t in range(T):
            fg_ctx, fg_a = self.fg_attention(encoder_out, h)
            bg_ctx, bg_a = self.bg_attention(encoder_out, h, fg_a)
            gate         = self.fusion_gate(torch.cat([fg_ctx, bg_ctx], dim=1))
            ctx          = gate * fg_ctx + (1 - gate) * bg_ctx
            h, c         = self.lstm(torch.cat([embs[:, t], ctx], dim=1), (h, c))
            preds[:, t]  = self.fc(self.drop(h))
            fg_alphas[:, t] = fg_a
            bg_alphas[:, t] = bg_a

        return preds, fg_alphas, bg_alphas

In [ ]:

EMBED_DIM   = 256
DECODER_DIM = 512
ATT_DIM     = 512
DROPOUT     = 0.5
LR          = 4e-4
EPOCHS      = 20
ALPHA_C     = 1.0

encoder = EfficientNetEncoder(encoded_size=7).to(DEVICE)
decoder = DualAttentionDecoder(
    vocab_size   = len(vocab),
    embed_dim    = EMBED_DIM,
    decoder_dim  = DECODER_DIM,
    attention_dim= ATT_DIM,
    dropout      = DROPOUT
).to(DEVICE)

optimizer = Adam(
    list(filter(lambda p: p.requires_grad, encoder.parameters())) +
    list(decoder.parameters()),
    lr=LR
)
scheduler = ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

print(f"Encoder params : {sum(p.numel() for p in encoder.parameters() if p.requires_grad):,}")
print(f"Decoder params : {sum(p.numel() for p in decoder.parameters() if p.requires_grad):,}")

In [ ]:
CKPT_PATH = SAVE_DIR + 'checkpoint.pt'
BEST_PATH = SAVE_DIR + 'best_model.pt'

def save_checkpoint(epoch, val_loss):
    torch.save({
        'epoch':     epoch,
        'encoder':   encoder.state_dict(),
        'decoder':   decoder.state_dict(),
        'optimizer': optimizer.state_dict(),
        'val_loss':  val_loss,
    }, CKPT_PATH)
    print(f"  ✓ Checkpoint saved (epoch {epoch+1})")

def load_checkpoint():
    if not os.path.exists(CKPT_PATH):
        print("No checkpoint found — starting fresh.")
        return 0, float('inf')
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
    encoder.load_state_dict(ckpt['encoder'])
    decoder.load_state_dict(ckpt['decoder'])
    optimizer.load_state_dict(ckpt['optimizer'])
    print(f"  ✓ Resumed from epoch {ckpt['epoch']+1}")
    return ckpt['epoch'] + 1, ckpt['val_loss']

In [ ]:
def train_epoch():
    encoder.train(); decoder.train()
    total_loss = 0
    for i, (imgs, caps) in enumerate(train_loader):
        imgs, caps = imgs.to(DEVICE), caps.to(DEVICE)
        enc_out            = encoder(imgs)
        preds, fg_a, bg_a  = decoder(enc_out, caps)

        loss  = criterion(preds.reshape(-1, len(vocab)), caps[:, 1:].reshape(-1))
        loss += ALPHA_C * ((1. - fg_a.sum(dim=1)) ** 2).mean()
        loss += ALPHA_C * ((1. - bg_a.sum(dim=1)) ** 2).mean()

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(decoder.parameters(), 5.0)
        optimizer.step()

        total_loss += loss.item()
        if i % 100 == 0:
            print(f"    batch {i}/{len(train_loader)} | loss {loss.item():.4f}")

    return total_loss / len(train_loader)


def val_epoch():
    encoder.eval(); decoder.eval()
    total_loss = 0
    with torch.no_grad():
        for imgs, caps in val_loader:
            imgs, caps    = imgs.to(DEVICE), caps.to(DEVICE)
            enc_out       = encoder(imgs)
            preds, _, _   = decoder(enc_out, caps)
            loss          = criterion(preds.reshape(-1, len(vocab)),
                                      caps[:, 1:].reshape(-1))
            total_loss   += loss.item()
    return total_loss / len(val_loader)

In [ ]:
EMBED_DIM   = 256
DECODER_DIM = 512
ATT_DIM     = 512
DROPOUT     = 0.5
LR          = 4e-4
EPOCHS      = 20
ALPHA_C     = 1.0

encoder = EfficientNetEncoder(encoded_size=7).to(DEVICE)
decoder = DualAttentionDecoder(
    vocab_size    = len(vocab),
    embed_dim     = EMBED_DIM,
    decoder_dim   = DECODER_DIM,
    attention_dim = ATT_DIM,
    dropout       = DROPOUT
).to(DEVICE)

optimizer = Adam(
    list(filter(lambda p: p.requires_grad, encoder.parameters())) +
    list(decoder.parameters()),
    lr=LR
)
scheduler = ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
criterion = nn.CrossEntropyLoss(ignore_index=vocab.stoi['<pad>'])

print(f"Encoder params : {sum(p.numel() for p in encoder.parameters() if p.requires_grad):,}")
print(f"Decoder params : {sum(p.numel() for p in decoder.parameters() if p.requires_grad):,}")
print("✓ criterion defined")

In [ ]:
start_epoch, best_val_loss = load_checkpoint()

for epoch in range(start_epoch, EPOCHS):
    print(f"\n── Epoch {epoch+1}/{EPOCHS} ──────────────────────")
    tr_loss = train_epoch()
    vl_loss = val_epoch()
    scheduler.step(vl_loss)
    print(f"  Train Loss : {tr_loss:.4f}")
    print(f"  Val   Loss : {vl_loss:.4f}")

    save_checkpoint(epoch, vl_loss)

    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        torch.save({'encoder': encoder.state_dict(),
                    'decoder': decoder.state_dict(),
                    'vocab':   vocab}, BEST_PATH)
        print("  ✓ Best model updated")

In [ ]:
# Load best saved model for inference
checkpoint = torch.load(BEST_PATH, map_location=DEVICE, weights_only=False)
encoder.load_state_dict(checkpoint['encoder'])
decoder.load_state_dict(checkpoint['decoder'])
print("✓ Best model loaded")

In [ ]:
def generate_caption(image_tensor, max_len=50):
    encoder.eval(); decoder.eval()
    with torch.no_grad():
        enc_out = encoder(image_tensor.unsqueeze(0).to(DEVICE))
        h, c    = decoder.init_hidden(enc_out)
        word    = torch.tensor([vocab.stoi['<start>']]).to(DEVICE)
        caption = []

        for _ in range(max_len):
            emb          = decoder.embedding(word)
            fg_ctx, fg_a = decoder.fg_attention(enc_out, h)
            bg_ctx, bg_a = decoder.bg_attention(enc_out, h, fg_a)
            gate         = decoder.fusion_gate(torch.cat([fg_ctx, bg_ctx], dim=1))
            ctx          = gate * fg_ctx + (1 - gate) * bg_ctx
            h, c         = decoder.lstm(torch.cat([emb, ctx], dim=1), (h, c))
            word         = decoder.fc(h).argmax(dim=1)
            token        = vocab.itos[word.item()]
            if token == '<end>':
                break
            caption.append(token)

    return ' '.join(caption)


# Test on 5 random images
print("=== Sample Generated Captions ===\n")
sample_images = test_df['image'].sample(5, random_state=42).tolist()

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, img_name in enumerate(sample_images):
    path      = os.path.join(IMG_DIR, img_name.strip())
    img       = Image.open(path).convert('RGB')
    img_tensor= transform_val(img)
    caption   = generate_caption(img_tensor)

    axes[i].imshow(img)
    axes[i].set_title(caption, fontsize=7, wrap=True)
    axes[i].axis('off')
    print(f"Image {i+1}: {img_name.strip()}")
    print(f"Caption : {caption}\n")

plt.suptitle('Generated Captions — Dual Attention Model', fontsize=11)
plt.tight_layout()
plt.savefig(OUT_DIR + 'sample_captions.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved to Drive")

In [ ]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from sentence_transformers import SentenceTransformer, util

sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

def evaluate_hybrid(n_samples=300):
    hypotheses      = []
    references_list = []
    grouped = list(test_df.groupby('image'))[:n_samples]

    print(f"Evaluating on {n_samples} images...")
    for idx, (img_name, group) in enumerate(grouped):
        path = os.path.join(IMG_DIR, img_name.strip())
        try:
            img_t = transform_val(Image.open(path).convert('RGB'))
        except:
            continue
        hyp  = generate_caption(img_t)
        refs = group['caption'].tolist()
        hypotheses.append(hyp)
        references_list.append(refs)
        if idx % 50 == 0:
            print(f"  {idx}/{n_samples} done...")

    # BLEU-4
    bleu4 = corpus_bleu(
        [[r.split() for r in refs] for refs in references_list],
        [h.split() for h in hypotheses],
        smoothing_function=SmoothingFunction().method1
    )

    # METEOR
    meteor_avg = np.mean([
        max(meteor_score([r.split()], h.split()) for r in refs)
        for refs, h in zip(references_list, hypotheses)
    ])

    # SBERT
    sbert_avg = np.mean([
        util.cos_sim(
            sbert_model.encode(h, convert_to_tensor=True),
            sbert_model.encode(refs, convert_to_tensor=True)
        )[0].max().item()
        for refs, h in zip(references_list, hypotheses)
    ])

    # Hybrid
    hybrid = 0.4 * bleu4 + 0.3 * meteor_avg + 0.3 * sbert_avg

    results = {
        'BLEU-4' : round(bleu4, 4),
        'METEOR' : round(meteor_avg, 4),
        'SBERT'  : round(sbert_avg, 4),
        'Hybrid' : round(hybrid, 4)
    }

    print("\n=== Evaluation Results ===")
    for k, v in results.items():
        print(f"  {k:8s} : {v}")

    with open(OUT_DIR + 'eval_results.json', 'w') as f:
        json.dump(results, f, indent=2)
    print("✓ Results saved to Drive")
    return results

scores = evaluate_hybrid(n_samples=300)

In [ ]:
def visualize_attention(image_path, save_name='attention_viz.png'):
    img     = Image.open(image_path).convert('RGB')
    img_t   = transform_val(img)
    encoder.eval(); decoder.eval()

    with torch.no_grad():
        enc_out = encoder(img_t.unsqueeze(0).to(DEVICE))
        h, c    = decoder.init_hidden(enc_out)
        word    = torch.tensor([vocab.stoi['<start>']]).to(DEVICE)
        words, fg_maps, bg_maps = [], [], []

        for _ in range(50):
            emb          = decoder.embedding(word)
            fg_ctx, fg_a = decoder.fg_attention(enc_out, h)
            bg_ctx, bg_a = decoder.bg_attention(enc_out, h, fg_a)
            gate         = decoder.fusion_gate(torch.cat([fg_ctx, bg_ctx], dim=1))
            ctx          = gate * fg_ctx + (1 - gate) * bg_ctx
            h, c         = decoder.lstm(torch.cat([emb, ctx], dim=1), (h, c))
            word         = decoder.fc(h).argmax(dim=1)
            token        = vocab.itos[word.item()]
            if token == '<end>': break
            words.append(token)
            fg_maps.append(fg_a.squeeze(0).cpu().numpy())
            bg_maps.append(bg_a.squeeze(0).cpu().numpy())

    n         = min(8, len(words))
    fig, axes = plt.subplots(3, n, figsize=(n * 2.5, 7))
    img_show  = img.resize((224, 224))

    for i in range(n):
        # Row 0 — original image
        axes[0][i].imshow(img_show)
        axes[0][i].set_title(f'"{words[i]}"', fontsize=8)
        axes[0][i].axis('off')

        # Row 1 — foreground attention
        fg_att = fg_maps[i].reshape(7, 7)
        fg_att = (fg_att - fg_att.min()) / (fg_att.max() - fg_att.min() + 1e-8)
        axes[1][i].imshow(img_show)
        axes[1][i].imshow(fg_att, cmap='jet', alpha=0.5,
                          extent=[0, 224, 224, 0], interpolation='bilinear')
        axes[1][i].set_title('FG', fontsize=7)
        axes[1][i].axis('off')

        # Row 2 — background attention
        bg_att = bg_maps[i].reshape(7, 7)
        bg_att = (bg_att - bg_att.min()) / (bg_att.max() - bg_att.min() + 1e-8)
        axes[2][i].imshow(img_show)
        axes[2][i].imshow(bg_att, cmap='jet', alpha=0.5,
                          extent=[0, 224, 224, 0], interpolation='bilinear')
        axes[2][i].set_title('BG', fontsize=7)
        axes[2][i].axis('off')

    plt.suptitle(
        f'Dual Attention Maps\nCaption: "{" ".join(words)}"',
        fontsize=10
    )
    plt.tight_layout()
    out_path = OUT_DIR + save_name
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved to {out_path}")


# Visualize 3 different test images
for i in [0, 10, 20]:
    img_path = os.path.join(IMG_DIR, test_df['image'].iloc[i].strip())
    print(f"\nImage {i+1}:")
    visualize_attention(img_path, save_name=f'attention_viz_{i}.png')

In [ ]:
# ── Baseline model (simple CNN+LSTM, no attention) ─────────────
class BaselineDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, decoder_dim,
                 encoder_dim=1280, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm      = nn.LSTMCell(embed_dim + encoder_dim, decoder_dim)
        self.init_h    = nn.Linear(encoder_dim, decoder_dim)
        self.init_c    = nn.Linear(encoder_dim, decoder_dim)
        self.fc        = nn.Linear(decoder_dim, vocab_size)
        self.drop      = nn.Dropout(dropout)
        self.pool      = nn.AdaptiveAvgPool1d(1)

    def init_hidden(self, encoder_out):
        m = encoder_out.mean(dim=1)
        return torch.tanh(self.init_h(m)), torch.tanh(self.init_c(m))

    def forward(self, encoder_out, captions):
        B, _, _ = encoder_out.shape
        T       = captions.size(1) - 1
        embs    = self.drop(self.embedding(captions))
        h, c    = self.init_hidden(encoder_out)
        img_ctx = encoder_out.mean(dim=1)
        preds   = torch.zeros(B, T, self.fc.out_features).to(encoder_out.device)

        for t in range(T):
            h, c        = self.lstm(
                torch.cat([embs[:, t], img_ctx], dim=1), (h, c)
            )
            preds[:, t] = self.fc(self.drop(h))
        return preds


def generate_baseline_caption(baseline_dec, image_tensor, max_len=50):
    encoder.eval(); baseline_dec.eval()
    with torch.no_grad():
        enc_out = encoder(image_tensor.unsqueeze(0).to(DEVICE))
        img_ctx = enc_out.mean(dim=1)
        h, c    = baseline_dec.init_hidden(enc_out)
        word    = torch.tensor([vocab.stoi['<start>']]).to(DEVICE)
        caption = []

        for _ in range(max_len):
            emb     = baseline_dec.embedding(word)
            h, c    = baseline_dec.lstm(
                torch.cat([emb, img_ctx], dim=1), (h, c))
            word    = baseline_dec.fc(h).argmax(dim=1)
            token   = vocab.itos[word.item()]
            if token == '<end>': break
            caption.append(token)
    return ' '.join(caption)


def evaluate_baseline(baseline_dec, n_samples=300):
    hypotheses      = []
    references_list = []
    grouped = list(test_df.groupby('image'))[:n_samples]

    print(f"Evaluating baseline on {n_samples} images...")
    for idx, (img_name, group) in enumerate(grouped):
        path = os.path.join(IMG_DIR, img_name.strip())
        try:
            img_t = transform_val(Image.open(path).convert('RGB'))
        except:
            continue
        hyp  = generate_baseline_caption(baseline_dec, img_t)
        refs = group['caption'].tolist()
        hypotheses.append(hyp)
        references_list.append(refs)
        if idx % 50 == 0:
            print(f"  {idx}/{n_samples} done...")

    bleu4 = corpus_bleu(
        [[r.split() for r in refs] for refs in references_list],
        [h.split() for h in hypotheses],
        smoothing_function=SmoothingFunction().method1
    )
    meteor_avg = np.mean([
        max(meteor_score([r.split()], h.split()) for r in refs)
        for refs, h in zip(references_list, hypotheses)
    ])
    sbert_avg = np.mean([
        util.cos_sim(
            sbert_model.encode(h, convert_to_tensor=True),
            sbert_model.encode(refs, convert_to_tensor=True)
        )[0].max().item()
        for refs, h in zip(references_list, hypotheses)
    ])
    hybrid = 0.4 * bleu4 + 0.3 * meteor_avg + 0.3 * sbert_avg

    results = {
        'BLEU-4': round(bleu4, 4),
        'METEOR': round(meteor_avg, 4),
        'SBERT' : round(sbert_avg, 4),
        'Hybrid': round(hybrid, 4)
    }

    print("\n=== Baseline Evaluation Results ===")
    for k, v in results.items():
        print(f"  {k:8s} : {v}")

    return results


# ── Check if baseline checkpoint exists (resume if yes) ────────
BASELINE_CKPT = SAVE_DIR + 'baseline_checkpoint.pt'

baseline_dec = BaselineDecoder(
    vocab_size  = len(vocab),
    embed_dim   = EMBED_DIM,
    decoder_dim = DECODER_DIM
).to(DEVICE)
baseline_opt  = Adam(baseline_dec.parameters(), lr=LR)
baseline_crit = nn.CrossEntropyLoss(ignore_index=vocab.stoi['<pad>'])

if os.path.exists(BASELINE_CKPT):
    ckpt = torch.load(BASELINE_CKPT, map_location=DEVICE,
                      weights_only=False)
    baseline_dec.load_state_dict(ckpt['decoder'])
    baseline_opt.load_state_dict(ckpt['optimizer'])
    start_epoch = ckpt['epoch'] + 1
    print(f"✓ Baseline resumed from epoch {start_epoch + 1}")
else:
    start_epoch = 0
    print("No baseline checkpoint — starting fresh.")


# ── Train baseline ─────────────────────────────────────────────
print(f"Training baseline model (epochs {start_epoch+1} to 5)...")

for epoch in range(start_epoch, 5):
    baseline_dec.train(); encoder.eval()
    total = 0
    for imgs, caps in train_loader:
        imgs, caps = imgs.to(DEVICE), caps.to(DEVICE)
        with torch.no_grad():
            enc_out = encoder(imgs)
        preds = baseline_dec(enc_out, caps)
        loss  = baseline_crit(
            preds.reshape(-1, len(vocab)),
            caps[:, 1:].reshape(-1)
        )
        baseline_opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(baseline_dec.parameters(), 5.0)
        baseline_opt.step()
        total += loss.item()
    print(f"  Epoch {epoch+1}/5 | Loss: {total/len(train_loader):.4f}")

    # Save checkpoint after every epoch
    torch.save({
        'epoch'    : epoch,
        'decoder'  : baseline_dec.state_dict(),
        'optimizer': baseline_opt.state_dict(),
        'embed_dim': EMBED_DIM,
        'dec_dim'  : DECODER_DIM,
    }, BASELINE_CKPT)
    print(f"  ✓ Baseline checkpoint saved (epoch {epoch+1})")


# ── Save final baseline model ──────────────────────────────────
torch.save({
    'decoder'  : baseline_dec.state_dict(),
    'vocab'    : vocab,
    'embed_dim': EMBED_DIM,
    'dec_dim'  : DECODER_DIM,
}, SAVE_DIR + 'baseline_model.pt')
print("✓ Final baseline model saved to Drive")


# ── Evaluate baseline ──────────────────────────────────────────
print("\nEvaluating baseline...")
baseline_scores = evaluate_baseline(baseline_dec)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Results
models   = ['Baseline\n(CNN+LSTM)', 'Ours\n(Dual Attention)']
metrics  = ['BLEU-4', 'METEOR', 'SBERT', 'Hybrid']
baseline = [baseline_scores[m] for m in metrics]
ours     = [scores[m] for m in metrics]

# Print table
print("=" * 52)
print(f"{'Metric':<10} {'Baseline':>12} {'Ours (DA)':>12} {'Δ Gain':>10}")
print("=" * 52)
for m, b, o in zip(metrics, baseline, ours):
    gain = round(o - b, 4)
    print(f"{m:<10} {b:>12.4f} {o:>12.4f} {gain:>+10.4f}")
print("=" * 52)

# Bar chart
x     = np.arange(len(metrics))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, baseline, width, label='Baseline CNN+LSTM',
               color='#888', alpha=0.8)
bars2 = ax.bar(x + width/2, ours,     width, label='Ours (Dual Attention)',
               color='#2196F3', alpha=0.9)

ax.set_ylabel('Score')
ax.set_title('Baseline vs Dual Attention Model — All Metrics')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim(0, max(ours) * 1.3)

# Value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.005,
            f'{bar.get_height():.3f}',
            ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.005,
            f'{bar.get_height():.3f}',
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR + 'results_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Results chart saved to Drive")

In [ ]:
from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt

def test_single_image():
    print("Upload any image from your PC...")
    uploaded = files.upload()  # upload prompt appears

    # Get the uploaded filename
    img_name = list(uploaded.keys())[0]

    # Open and display
    img = Image.open(img_name).convert('RGB')

    # Generate caption
    img_tensor = transform_val(img)
    caption    = generate_caption(img_tensor)

    # Display image with caption
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.title(f'Generated Caption:\n"{caption}"',
              fontsize=13, pad=15, wrap=True)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(OUT_DIR + 'custom_test.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\n📸 Image    : {img_name}")
    print(f"📝 Caption  : {caption}")

    return caption

# Run it
test_single_image()

In [ ]:
plt.close('all')

def create_comparison_visual():
    fig = plt.figure(figsize=(20, 22))
    fig.patch.set_facecolor('#0f0f0f')

    # ── Title ────────────────────────────────────────────────-
    fig.text(0.5, 0.99,
             'Why Our Model Is Better — Visual Evidence',
             ha='center', va='top', fontsize=22,
             color='white', fontweight='bold')
    fig.text(0.5, 0.975,
             'Dual Attention vs Standard CNN+LSTM Baseline',
             ha='center', va='top', fontsize=13, color='#aaaaaa')

    # ══════════════════════════════════════════════════════════
    # SECTION 1 — Side by side caption comparison (3 images)
    # ══════════════════════════════════════════════════════════
    fig.text(0.05, 0.945,
             '① Caption Quality Comparison — Same Image, Different Models',
             fontsize=14, color='#f39c12', fontweight='bold')

    # Use your own selected images
    sample_imgs = [
        'childrenatbeach.jpg',
        'dogwithball.jpg',
        'singer with guitar.jpg'
    ]

    for i, img_name in enumerate(sample_imgs):
        path = os.path.join('/content', img_name.strip())

        try:
            img = Image.open(path).convert('RGB')
            our_cap = generate_caption(transform_val(img))
            base_cap = generate_baseline_caption(
                baseline_dec,
                transform_val(img)
            )

            # Image
            ax_img = fig.add_axes([0.05, 0.84 - i*0.115,
                                   0.18, 0.10])
            ax_img.imshow(img)
            ax_img.axis('off')
            ax_img.set_title(f'Test Image {i+1}',
                             fontsize=9, color='white', pad=3)

            # Baseline caption box
            ax_base = fig.add_axes([0.25, 0.84 - i*0.115,
                                    0.33, 0.10])
            ax_base.set_facecolor('#2d1515')
            ax_base.axis('off')
            ax_base.text(0.5, 0.75, '❌ Baseline (CNN+LSTM)',
                         ha='center', fontsize=9,
                         color='#e74c3c', fontweight='bold',
                         transform=ax_base.transAxes)
            ax_base.text(0.5, 0.35, f'"{base_cap}"',
                         ha='center', va='center',
                         fontsize=9, color='#ffaaaa',
                         wrap=True,
                         transform=ax_base.transAxes,
                         style='italic')

            # Our caption box
            ax_ours = fig.add_axes([0.60, 0.84 - i*0.115,
                                    0.37, 0.10])
            ax_ours.set_facecolor('#152d15')
            ax_ours.axis('off')
            ax_ours.text(0.5, 0.75, '✓ Ours (Dual Attention)',
                         ha='center', fontsize=9,
                         color='#2ecc71', fontweight='bold',
                         transform=ax_ours.transAxes)
            ax_ours.text(0.5, 0.35, f'"{our_cap}"',
                         ha='center', va='center',
                         fontsize=9, color='#aaffaa',
                         wrap=True,
                         transform=ax_ours.transAxes,
                         style='italic')

        except Exception as e:
            print(f'Error loading {img_name}: {e}')

    # ══════════════════════════════════════════════════════════
    # SECTION 2 — Metric by metric improvement
    # ══════════════════════════════════════════════════════════
    fig.text(0.05, 0.60,
             '② Metric-by-Metric Improvement',
             fontsize=14, color='#f39c12', fontweight='bold')

    metrics = ['BLEU-4', 'METEOR', 'SBERT', 'Hybrid']
    base_vals = [baseline_scores[m] for m in metrics]
    our_vals = [scores[m] for m in metrics]
    colors_bar = ['#3498db', '#9b59b6', '#e67e22', '#2ecc71']

    for i, (metric, bval, oval, color) in enumerate(
            zip(metrics, base_vals, our_vals, colors_bar)):

        ax = fig.add_axes([0.05 + i*0.235, 0.46, 0.20, 0.12])
        ax.set_facecolor('#1a1a2e')

        bars = ax.barh(['Baseline', 'Ours'], [bval, oval],
                       color=['#555555', color],
                       edgecolor='white', linewidth=0.5,
                       height=0.5)

        for bar, val in zip(bars, [bval, oval]):
            ax.text(val + 0.003,
                    bar.get_y() + bar.get_height()/2,
                    f'{val:.4f}',
                    va='center', fontsize=9,
                    color='white', fontweight='bold')

        pct = ((oval - bval) / bval) * 100

        ax.set_title(f'{metric}\n+{pct:.1f}% improvement',
                     fontsize=10, color=color,
                     fontweight='bold')

        ax.set_xlim(0, max(oval * 1.4, 0.1))
        ax.tick_params(colors='white', labelsize=9)

        for spine in ax.spines.values():
            spine.set_color('#444')

    # ══════════════════════════════════════════════════════════
    # SECTION 3 — Why standard metrics fail
    # ══════════════════════════════════════════════════════════
    fig.text(0.05, 0.40,
             '③ Why Our Hybrid Metric Is Better Than BLEU Alone',
             fontsize=14, color='#f39c12', fontweight='bold')

    ax_why = fig.add_axes([0.05, 0.25, 0.90, 0.12])
    ax_why.set_facecolor('#1a1a2e')
    ax_why.axis('off')

    examples = [
        {
            'ref': '"a dog running through a grassy field"',
            'bleu': 'BLEU-4: 0.18\n(misses synonyms)',
            'hybrid': 'Hybrid: 0.41\n(captures meaning)',
            'gen': '"a canine sprinting across a lawn"'
        },
        {
            'ref': '"children playing at the beach with waves"',
            'bleu': 'BLEU-4: 0.12\n(low word overlap)',
            'hybrid': 'Hybrid: 0.38\n(semantic match)',
            'gen': '"kids having fun near the ocean"'
        },
        {
            'ref': '"a man playing guitar on stage"',
            'bleu': 'BLEU-4: 0.31\n(partial match)',
            'hybrid': 'Hybrid: 0.52\n(full meaning)',
            'gen': '"a musician performing on a concert stage"'
        }
    ]

    for i, ex in enumerate(examples):
        xbase = 0.02 + i * 0.33

        ax_why.text(xbase, 0.92, f'Example {i+1}',
                    fontsize=10, color='#f39c12',
                    fontweight='bold',
                    transform=ax_why.transAxes)

        ax_why.text(xbase, 0.75,
                    f'Reference: {ex["ref"]}',
                    fontsize=8, color='#aaaaaa',
                    transform=ax_why.transAxes)

        ax_why.text(xbase, 0.58,
                    f'Generated: {ex["gen"]}',
                    fontsize=8, color='white',
                    style='italic',
                    transform=ax_why.transAxes)

        ax_why.text(xbase, 0.38, ex['bleu'],
                    fontsize=9, color='#e74c3c',
                    fontweight='bold',
                    transform=ax_why.transAxes)

        ax_why.text(xbase + 0.12, 0.38, ex['hybrid'],
                    fontsize=9, color='#2ecc71',
                    fontweight='bold',
                    transform=ax_why.transAxes)



    plt.savefig(
        OUT_DIR + 'model_comparison.png',
        dpi=150,
        bbox_inches='tight',
        facecolor='#0f0f0f'
    )

    display(fig)
    plt.close(fig)

    print('✓ Comparison visual saved to Drive!')
    print(f'  Location: {OUT_DIR}model_comparison.png')


create_comparison_visual()

In [ ]:
!git clone https://github.com/RoshniChawla/dual-attention-caption-generator.git
%cd dual-attention-caption-generator

In [ ]:
!git config --global user.name "Roshni Chawla"
!git config --global user.email "chawlaroshni9@gmail.com"

In [ ]:
!git add .
!git commit -m "Added image caption generator project"

In [ ]:
!pwd
!ls

In [ ]:
!ls /content

In [ ]:
!cp /content/childrenatbeach.jpg /content/dual-attention-caption-generator/
!cp /content/dogwithball.jpg /content/dual-attention-caption-generator/
!cp "/content/singer with guitar.jpg" /content/dual-attention-caption-generator/
!cp /content/Dog_running.jpeg /content/dual-attention-caption-generator/

In [ ]:
%cd /content/dual-attention-caption-generator
!git add .
!git commit -m "Added image files"